In [1]:
import os
output = "data/outputs/doc-as-input"
os.listdir(output)

['enko', 'wrong', 'enzh']

In [3]:
import json

def read_jsonl(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

wrong = os.path.join(output, "wrong")

In [4]:
files = [f for f in os.listdir(wrong) if f.endswith(".jsonl")]
d1 = read_jsonl(os.path.join(wrong, files[0]))

In [24]:
# output_token이 최대인 세그먼트 고르기
from collections import defaultdict


def get_max_segment(data: list):
    d1_dic = defaultdict(list)
    for l in data:
        d1_dic[(l["doc_id"], l["system"])].append(l)

    for k, v in d1_dic.items():
        token_len = [l["usage"]["output_token"] for l in v]
        max_idx = token_len.index(max(token_len))
        new_line = v[max_idx]
        new_line["ref_seg"] = ""
        new_line["human_pe_seg"] = ""
        new_line["seg_id"] = 0
        d1_dic[k] = new_line

    return d1_dic


In [25]:
for file in files:
    fdir = os.path.join(wrong, file)
    data = read_jsonl(fdir)
    new_data =get_max_segment(data)
    print(file, len(new_data))

    with open(os.path.join(output, "enko", file), "w", encoding="utf-8") as f:
        for v in new_data.values():
            f.write(json.dumps(v, ensure_ascii=False) + "\n")

wmt24pp_tower-plus-72b.jsonl 472
wmt24pp_gpt-5.4-2026-03-05.jsonl 472
wmt25_hunyuan-mt-7b.jsonl 900
wmt24pp_hunyuan-mt-7b.jsonl 472


In [19]:
next(iter(d1_dic))

('test-en-news_beverly_press.3585', 'Gemini-1')

In [20]:
d1_dic[('test-en-news_beverly_press.3585', 'Gemini-1')]

{'seg_id': 1,
 'doc_id': 'test-en-news_beverly_press.3585',
 'bucket_id': 'medium',
 'system': 'Gemini-1',
 'ref_seg': '신규 갤러리 전시의 주인공은 땅과 물을 담은 시소의 작품',
 'human_pe_seg': '신규 갤러리 전시의 주인공은 땅과 물을 담은 시소의 작품',
 'mt_pe_seg': '<pe>티에라 델 솔은 웨스트 할리우드의 새로운 갤러리에서 "비센테 시소: 땅과 물의 기억" 전시를 선보이게 되어 기쁩니다. 시소는 2012년부터 스튜디오 아트 프로그램 소속 작가로 활동해 왔으며, 이번 전시는 그의 첫 개인전입니다. 시소는 1962년 마드리드에서 태어나 베네수엘라, 트리니다드, 마이애미를 오가며 성장했고, 20대 초반에 가족과 함께 남부 캘리포니아로 이주했습니다.\n\n다양한 소재를 능숙하게 다루는 시소는 아크릴, 파스텔, 연필, 수채화 등으로 풍부한 풍경화, 초상화, 정물화 시리즈를 제작했습니다. 가족사진, 자신의 참고 사진, 기억을 바탕으로 한 그의 다채로운 작품들은 다양한 매체에 걸친 그의 관심과 기술의 폭을 보여줍니다. 시소의 열대 풍경화와 해양화는 그의 과거 지리적 배경을 반영하며, 풍부한 패턴을 사용하고 사람들을 등장시켜 문화, 기억, 환경 사이의 의미있는 연결을 보여줍니다. 시소는 자신의 작품에 스페인어와 영어를 혼용하여 제목을 붙이는데, 이는 로스앤젤레스 카운티에서의 그의 삶의 다채롭고 중요한 복합성을 나타냅니다. "비센테 시소: 땅과 물의 기억" 전시는 1월 13일 토요일 오후 6시부터 8시까지 오프닝 리셉션과 함께 시작하며, 3월 3일 일요일까지 전시됩니다.\n\n티에라 델 솔 갤러리는 산타 모니카 블러바드 7414번지에 위치해 있습니다. 자세한 정보는 tierradelsolgallery.org를 방문하세요.</pe>',
 'level': 'doc-as-input',
 'usage': {'input_token': 1592, 'output

# For retrieval

In [1]:
in_root = "data/wmt25/en-ko_KR/"
out_root = "data/wmt25/preliminary"

import os
import json
from script.utils import read_jsonl

In [2]:
data = read_jsonl(os.path.join(in_root, "input.jsonl"))

In [3]:
data[0]

{'seg_id': 0,
 'new_doc_id': 0,
 'doc_id': 'rink_rats_chapter1',
 'domain': 'literary',
 'system': 'Llama-4-Maverick',
 'src_lang': 'English',
 'tgt_lang': 'Korean (South Korea)',
 'src_seg': 'Rink Rats\nChapter 1: First Day\nKyle looked at his reflection in the mirror. He studied the lines etched on his face, each one a reminder of the years gone by. A hint of gray threaded his hair now, and for a moment, he felt the weight of change settling in. \nHe didn’t want to label it an identity crisis. Not yet at least. But as he splashed the cold sink water on his face, memories of his pro hockey career flooded his mind. The thrill of the game, the adrenaline of competition, the love of the fans. All forty years of his life had practically been dedicated to making it to the highest level. Now those days were behind him. ',
 'tgt_seg': '링크 래츠\n1장: 첫날\n카일은 거울 속 자신의 모습을 바라보았다. 얼굴에 새겨진 주름을 유심히 살피니, 그 하나하나가 세월의 흔적임을 새삼 느꼈다. 머리에는 희끗희끗 흰머리가 섞이기 시작했고, 한순간 그는 변화의 무게를 실감했다.\n자신의 정체성 위기를 확인하고 싶지는 않았다. 

In [4]:
from collections import defaultdict

per_sys = defaultdict(lambda: defaultdict(list))

for line in data:
    seg = {"src": line["src_seg"],  "tgt": line["tgt_seg"],}
    per_sys[line["system"]][line["doc_id"]].append(seg)

In [8]:
next(iter(per_sys))

'Llama-4-Maverick'

In [10]:
len(per_sys["Llama-4-Maverick"])

25

In [11]:
from script.utils import write_jsonl

In [16]:
out_dir = os.path.join(out_root, "input")
os.makedirs(out_dir, exist_ok=True)

for sys, record in per_sys.items():
    with open(os.path.join(out_dir, f"{sys}.jsonl"), "w", encoding="utf-8") as f:
        for doc_id, segs in record.items():
            f.write(json.dumps({"doc_id": doc_id, "segments": segs}, ensure_ascii=False) + "\n")


## 모델 하나 고르기?

In [18]:
sample = read_jsonl(os.path.join(out_root, "output/k3", "Algharb.jsonl"))
len(sample)

265

In [19]:
sample[0]

{'doc_id': 'rink_rats_chapter1',
 'seg_idx': 0,
 'src_seg': 'Rink Rats\nChapter 1: First Day\nKyle looked at his reflection in the mirror. He studied the lines etched on his face, each one a reminder of the years gone by. A hint of gray threaded his hair now, and for a moment, he felt the weight of change settling in. \nHe didn’t want to label it an identity crisis. Not yet at least. But as he splashed the cold sink water on his face, memories of his pro hockey career flooded his mind. The thrill of the game, the adrenaline of competition, the love of the fans. All forty years of his life had practically been dedicated to making it to the highest level. Now those days were behind him. ',
 'tgt_seg': '링크 래츠\n제 1장: 첫째 날\n카일은 거울에 비친 자신의 모습을 바라보았다. 그는 얼굴에 새겨진 주름 하나하나를 유심히 살펴보았고, 각각은 지나간 세월을 상기시키는 흔적이었다. 이제 그의 머리카락에는 희끗희끗한 회색 가닥이 섞여 있었고, 잠시 동안 그는 서서히 다가오는 변화의 무게를 느꼈다.\n그는 그것을 정체성 위기라고 규정하고 싶지 않았다. 적어도 아직은 아니었다. 하지만 싱크대의 차가운 물을 얼굴에 튀기면서, 그의 프로 하키 선수 시절의 기억들이 머릿속을 가득 채웠다. 경기에 대한 짜릿함, 경쟁에서 느끼는

## TER 계산하기

In [35]:
ter_metric = TER_Metric()

def calc_ter(line):
    hyp = line["mt_pe_seg"].replace("<pe>", "").replace("</pe>", "")
    scores = ter_metric.evaluate([{"tgt_seg": hyp, "ref_seg": line["tgt_seg"]}])
    scores = [min(s, 100) if s < 100 else None for s in scores]
    return scores

In [54]:
import os
from docape.utils import read_jsonl
from collections import defaultdict

fdir = "data/outputs/rag/preliminary/{level}"

for level in ("high", "low"):
    for k in (1, 3, 5, 10):
        data = read_jsonl(os.path.join(fdir.format(level=level), f"wmt25_Gemini-2_{k}.jsonl"))
        with open(os.path.join(fdir.format(level=level), f"wmt25_Gemini-2_{k}.jsonl"), "w", encoding="utf-8") as f:
            for line in data:
                line.setdefault("scores", {})["ter"] = calc_ter(line)[0]
                f.write(json.dumps(line, ensure_ascii=False) + "\n")


In [37]:
for k, items in result.items():
    n_high = items["high"].count(None)
    n_low = items["low"].count(None)
    print(f"k={k}: high None count={n_high}, low None count={n_low}")

k=1: high None count=3, low None count=7
k=3: high None count=8, low None count=4
k=5: high None count=3, low None count=4
k=10: high None count=11, low None count=4


In [38]:
items["high"]

[None,
 0.0,
 2.5316455696202533,
 17.431192660550458,
 29.545454545454547,
 20.652173913043477,
 9.090909090909092,
 6.976744186046512,
 27.631578947368425,
 22.78481012658228,
 87.8048780487805,
 9.433962264150944,
 98.07692307692307,
 13.186813186813188,
 6.741573033707865,
 18.269230769230766,
 23.61111111111111,
 82.0,
 96.07843137254902,
 76.71232876712328,
 89.58333333333334,
 23.711340206185564,
 18.367346938775512,
 7.865168539325842,
 10.1010101010101,
 94.44444444444444,
 10.38961038961039,
 9.523809523809524,
 88.88888888888889,
 31.132075471698112,
 84.14634146341463,
 30.18867924528302,
 22.448979591836736,
 16.470588235294116,
 5.88235294117647,
 90.81632653061224,
 35.714285714285715,
 96.26168224299066,
 5.681818181818182,
 28.26086956521739,
 2.0408163265306123,
 13.513513513513514,
 0.0,
 None,
 None,
 None,
 45.67901234567901,
 94.44444444444444,
 89.47368421052632,
 92.85714285714286,
 14.084507042253522,
 1.1627906976744187,
 9.67741935483871,
 34.375,
 97.9166666

In [49]:
import numpy as np

final = dict()
for k, items in result.items():
    high = [s if s is not None else 100.0 for s in items["high"] ]
    low  = [s if s is not None else 100.0 for s in items["low"] ]
    delta = np.abs(np.array(high) - np.array(low))
    final[k] = delta


TypeError: only length-1 arrays can be converted to Python scalars

[100,
 0.0,
 2.5316455696202533,
 17.431192660550458,
 29.545454545454547,
 20.652173913043477,
 9.090909090909092,
 6.976744186046512,
 27.631578947368425,
 22.78481012658228,
 87.8048780487805,
 9.433962264150944,
 98.07692307692307,
 13.186813186813188,
 6.741573033707865,
 18.269230769230766,
 23.61111111111111,
 82.0,
 96.07843137254902,
 76.71232876712328,
 89.58333333333334,
 23.711340206185564,
 18.367346938775512,
 7.865168539325842,
 10.1010101010101,
 94.44444444444444,
 10.38961038961039,
 9.523809523809524,
 88.88888888888889,
 31.132075471698112,
 84.14634146341463,
 30.18867924528302,
 22.448979591836736,
 16.470588235294116,
 5.88235294117647,
 90.81632653061224,
 35.714285714285715,
 96.26168224299066,
 5.681818181818182,
 28.26086956521739,
 2.0408163265306123,
 13.513513513513514,
 0.0,
 100,
 100,
 100,
 45.67901234567901,
 94.44444444444444,
 89.47368421052632,
 92.85714285714286,
 14.084507042253522,
 1.1627906976744187,
 9.67741935483871,
 34.375,
 97.91666666666

In [9]:
scores[:5]

[[94.52054794520548],
 [48.35164835164835],
 [29.11392405063291],
 [44.03669724770643],
 [97.72727272727273]]